In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import pandas as pd


In [2]:
X, y = load_breast_cancer(return_X_y=True)          # features, labels

# hold out 20% as a test set the model NEVER sees
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)


In [3]:

for depth in [1, 3, 5, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)                    # learn on TRAIN
    train = model.score(X_train, y_train)          # score on TRAIN
    test  = model.score(X_test,  y_test)           # score on TEST
    print(depth, round(train, 3), round(test, 3))

1 0.923 0.921
3 0.976 0.939
5 0.993 0.921
None 1.0 0.912


In [4]:
model.score(X_test, y_test)

0.9122807017543859

In [4]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

X, y = load_breast_cancer(return_X_y=True)
model = LogisticRegression(max_iter=5000)

# cv=5 splits, trains & tests 5 times
scores = cross_val_score(model, X, y, cv=5)

print(scores)         # [0.939 0.947 0.982 0.930 0.956]
print(scores.mean())  # 0.951
print(scores.std())   # 0.018

/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: Runti

[0.93859649 0.94736842 0.98245614 0.92982456 0.95575221]
0.9507995652848935
0.01804054330253301


/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site

In [6]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline

# WRONG: pick features using ALL the data, THEN cross-validate
X_sel = SelectKBest(f_classif, k=20).fit_transform(X, y)          # leak!
leaky = cross_val_score(LogisticRegression(max_iter=1000), X_sel, y, cv=5)

# RIGHT: selection lives INSIDE the pipeline, re-run per fold
pipe = Pipeline([
    ("select", SelectKBest(f_classif, k=20)),
    ("clf",    LogisticRegression(max_iter=1000)),
])
honest = cross_val_score(pipe, X, y, cv=5)                        # no leak
print(leaky.mean(), honest.mean())   # 0.860   0.415

/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: Runti

0.9525694767893185 0.9525694767893185


/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/kbshal/mero_space/sarathi_acad

In [12]:
import numpy as np
import pandas as pd
n_neg = 980
n_pos = 20
np.random.seed(42)
neg_x1 = np.random.normal(loc=0, scale=1.5, size=n_neg)
neg_x2 = np.random.normal(loc=0, scale=1.5, size=n_neg)

pos_x1 = np.random.normal(loc=0.8, scale=1.5, size=n_pos)
pos_x2 = np.random.normal(loc=0.8, scale=1.5, size=n_pos)

#Two features: 
X1 = np.concatenate([neg_x1, pos_x1])
X2 = np.concatenate([neg_x2, pos_x2])
#Label:take the list [0] and repeat it 980
y = np.array([0]*n_neg + [1]*n_pos)
#[980 zeros]  +  [20 ones]  =  [980 zeros, then 20 ones]  (1000 items total)

df = pd.DataFrame({"feature_1": X1, "feature_2": X2, "label": y})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

df["label"].value_counts()

X = df[["feature_1","feature_2"]]
y = df["label"]
X.head()
y.head()

0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64

In [13]:
df

,feature_1,feature_2,label
0,0.815040,-0.826779,0
1,1.474036,0.285750,0
2,-2.761311,0.769659,0
3,-0.860493,0.036329,0
4,-1.686963,-1.016573,0
...,...,...,...
995,2.829279,1.545425,0
996,2.161910,0.496320,0
997,0.304385,-0.192806,0
998,0.111142,1.391760,0


In [14]:
X = df[["feature_1","feature_2"]]
y = df["label"]
X.head()
y.head()

0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64

In [ ]:
forest = RandomForestClassifier(n_estimators=200, random_state=42)
forest.fit(df[feature_1], y_train)
print(forest.score(X_train, y_train))   # 1.000
print(forest.score(X_test,  y_test))    # 0.956

In [29]:
import numpy as np
import pandas as pd

# ============================================================
# Build an imbalanced synthetic dataset (980 negative vs 20 positive)
# ============================================================
n_neg = 980
n_pos = 20
np.random.seed(42)
neg_x1 = np.random.normal(loc=0, scale=1.5, size=n_neg)
neg_x2 = np.random.normal(loc=0, scale=1.5, size=n_neg)
pos_x1 = np.random.normal(loc=0.8, scale=1.5, size=n_pos)
pos_x2 = np.random.normal(loc=0.8, scale=1.5, size=n_pos)

# Two features:
X1 = np.concatenate([neg_x1, pos_x1])
X2 = np.concatenate([neg_x2, pos_x2])

# Label: take the list [0] and repeat it 980
# [980 zeros] + [20 ones] = [980 zeros, then 20 ones] (1000 items total)
y = np.array([0] * n_neg + [1] * n_pos)

df = pd.DataFrame({"feature_1": X1, "feature_2": X2, "label": y})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle
#print(df["label"].value_counts())

X = df[["feature_1", "feature_2"]]
y = df["label"]
#print(X.head())
#print(y.head())

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


forest = RandomForestClassifier(n_estimators=200, random_state=42, max_depth=5)
forest.fit(X_train, y_train)

pred = forest.predict(X_test)

train_score = forest.score(X_train, y_train)
test_score = forest.score(X_test, y_test)
print("Train score:", train_score)
print("Test score: ", test_score)
print("======================")
cm = confusion_matrix(y_test, pred)
print(cm)
print("==================")
from sklearn.metrics import recall_score, f1_score
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred)
print(round(recall, 3),round(f1, 3))


#print(classification_report(y_test, pred))

Train score: 0.98375
Test score:  0.98
[[196   0]
 [  4   0]]
0.0 0.0


In [20]:
y

0      0
1      0
2      0
3      0
4      0
      ..
995    0
996    0
997    0
998    0
999    0
Name: label, Length: 1000, dtype: int64

In [21]:
X

,feature_1,feature_2
0,0.815040,-0.826779
1,1.474036,0.285750
2,-2.761311,0.769659
3,-0.860493,0.036329
4,-1.686963,-1.016573
...,...,...
995,2.829279,1.545425
996,2.161910,0.496320
997,0.304385,-0.192806
998,0.111142,1.391760


In [23]:
y[y==1].count()

np.int64(20)

In [25]:
y[y==0].count()

np.int64(980)

In [26]:
y_train[y_train==1].count()

np.int64(16)

In [27]:
y_test[y_test==1].count()

np.int64(4)

In [28]:
y_test[y_test==0].count()

np.int64(196)